# Speech to Minutes App - Tutorial Notebook

Questo notebook implementa un'applicazione Streamlit per convertire file audio in minute di meeting utilizzando:
- **🤗 Hugging Face Transformers** per la trascrizione speech-to-text (Whisper)
- **🤖 DeepSeek R1** per la generazione intelligente delle minute

## Architettura
1. 📁 Lettura file audio da cartella o upload
2. 🎵 Trascrizione con modelli Whisper da Hugging Face
3. 🤖 Generazione minute con DeepSeek R1 API
4. 📊 Interfaccia Streamlit per visualizzazione e download

## 1. Import Libraries and Setup

Importiamo le librerie necessarie e configuriamo l'ambiente per utilizzare i modelli Hugging Face.

In [ ]:
# Step 1: Import Libraries and Setup
print("🤗 STEP 1: Import Libraries and Setup")
print("-" * 50)

import os
import sys
import json
import time
import warnings
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Optional, Any
import traceback

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Core ML libraries - Hugging Face
try:
    from transformers import (
        pipeline, 
        AutoTokenizer, 
        AutoModelForCausalLM,
        AutoModelForSpeechSeq2Seq,
        AutoProcessor
    )
    import torch
    print("✅ Transformers e PyTorch caricati con successo")
    HF_AVAILABLE = True
except ImportError as e:
    print(f"❌ Errore import Transformers: {e}")
    print("   Installa con: pip install transformers torch")
    HF_AVAILABLE = False

# Audio processing libraries
try:
    import librosa
    import soundfile as sf
    print("✅ Librerie audio caricate")
    AUDIO_AVAILABLE = True
except ImportError as e:
    print(f"❌ Errore import audio libraries: {e}")
    print("   Installa con: pip install librosa soundfile")
    AUDIO_AVAILABLE = False

# Data processing
try:
    import numpy as np
    import pandas as pd
    print("✅ Librerie data processing caricate")
except ImportError as e:
    print(f"❌ Errore import data libraries: {e}")
    print("   Installa con: pip install numpy pandas")

# Device configuration per ottimizzazioni
if HF_AVAILABLE:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
    print(f"💻 Device: {DEVICE}")
    
    if DEVICE == "cuda":
        print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
        print(f"🔥 VRAM disponibile: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
else:
    DEVICE = "cpu"
    TORCH_DTYPE = "float32"

# Project paths
PROJECT_PATH = Path.cwd()
MODELS_CACHE_DIR = PROJECT_PATH / "models_cache"
MODELS_CACHE_DIR.mkdir(exist_ok=True)

print(f"📂 Directory progetto: {PROJECT_PATH}")
print(f"🗂️ Cache modelli: {MODELS_CACHE_DIR}")

# Configuration for models
MODEL_CONFIG = {
    "whisper": {
        "model_name": "openai/whisper-small",
        "alternatives": {
            "tiny": "openai/whisper-tiny",
            "base": "openai/whisper-base", 
            "small": "openai/whisper-small",
            "medium": "openai/whisper-medium",
            "large": "openai/whisper-large-v3"
        }
    },
    "text_generation": {
        "model_name": "microsoft/DialoGPT-medium", 
        "alternatives": {
            "small": "microsoft/DialoGPT-small",
            "medium": "microsoft/DialoGPT-medium",
            "large": "microsoft/DialoGPT-large",
            "multilingual": "Helsinki-NLP/opus-mt-en-it"
        }
    }
}

print("📋 Configurazione modelli:")
for category, config in MODEL_CONFIG.items():
    print(f"   {category}: {config['model_name']}")

if HF_AVAILABLE and AUDIO_AVAILABLE:
    print("\n✅ Tutti i sistemi pronti per l'elaborazione!")
else:
    print("\n⚠️ Alcune dipendenze mancanti - installa i pacchetti richiesti")

## 2. Setup Hugging Face Whisper Model

Configuriamo e carichiamo il modello Whisper da Hugging Face per la trascrizione audio.

In [ ]:
# Step 2: Classe Trascrittore Whisper (Hugging Face)
print("🎤 STEP 2: Whisper Speech-to-Text")
print("-" * 50)

class HuggingFaceWhisperTranscriber:
    """Trascrittore Whisper completamente locale usando Hugging Face."""
    
    def __init__(self, model_name: str = "openai/whisper-small"):
        if not HF_AVAILABLE:
            raise ImportError("Transformers non disponibile")
            
        self.model_name = model_name
        self.device = DEVICE
        self.pipe = None
        
        # Cache dei modelli disponibili
        self.available_models = MODEL_CONFIG["whisper"]["alternatives"]
        
        self._load_model()
    
    def _load_model(self):
        """Carica il modello Whisper da Hugging Face."""
        try:
            print(f"🤗 Caricamento modello Whisper: {self.model_name}")
            start_time = time.time()
            
            # Usa pipeline ottimizzato per speech recognition
            self.pipe = pipeline(
                "automatic-speech-recognition",
                model=self.model_name,
                tokenizer=self.model_name,
                return_timestamps=True,
                device=0 if self.device == "cuda" else -1,
                torch_dtype=TORCH_DTYPE,
                cache_dir=str(MODELS_CACHE_DIR)
            )
            
            load_time = time.time() - start_time
            print(f"✅ Modello caricato in {load_time:.2f} secondi")
            
        except Exception as e:
            print(f"❌ Errore caricamento modello: {e}")
            raise
    
    def transcribe_audio(self, audio_file: Path, language: str = "italian") -> Dict[str, Any]:
        """Trascrive un file audio con timestamps."""
        try:
            start_time = time.time()
            print(f"🎵 Trascrizione: {audio_file.name}")
            
            # Carica e preprocessa audio
            if not AUDIO_AVAILABLE:
                raise ImportError("Librerie audio non disponibili")
                
            # Usa librosa per caricare con sample rate ottimizzato per Whisper
            audio_data, sample_rate = librosa.load(str(audio_file), sr=16000)
            audio_duration = len(audio_data) / sample_rate
            
            print(f"📊 Audio info: {audio_duration:.2f}s, {sample_rate}Hz")
            
            # Trascrizione con Whisper
            print("🤖 Elaborazione con Whisper...")
            result = self.pipe(
                audio_data,
                return_timestamps=True,
                generate_kwargs={"language": language[:2] if language else "en"}
            )
            
            processing_time = time.time() - start_time
            word_count = len(result["text"].split())
            
            print(f"✅ Trascrizione completata: {word_count} parole in {processing_time:.2f}s")
            
            return {
                "success": True,
                "data": {
                    "text": result["text"].strip(),
                    "chunks": result.get("chunks", []),
                    "processing_time_seconds": processing_time,
                    "audio_duration_seconds": audio_duration,
                    "word_count": word_count,
                    "words_per_second": word_count / processing_time if processing_time > 0 else 0,
                    "model_used": self.model_name,
                    "timestamp": datetime.now().isoformat(),
                    "language": language,
                    "sample_rate": sample_rate
                },
                "error": None
            }
            
        except Exception as e:
            print(f"❌ Errore trascrizione: {e}")
            return {
                "success": False,
                "data": None,
                "error": str(e)
            }
    
    def list_available_models(self):
        """Lista i modelli Whisper disponibili."""
        print("📋 Modelli Whisper disponibili:")
        for size, model_name in self.available_models.items():
            print(f"   {size}: {model_name}")
    
    def change_model(self, new_model_name: str):
        """Cambia il modello Whisper."""
        try:
            print(f"🔄 Cambio modello: {self.model_name} → {new_model_name}")
            self.model_name = new_model_name
            self._load_model()
            print("✅ Modello cambiato con successo")
        except Exception as e:
            print(f"❌ Errore cambio modello: {e}")

# Test del trascrittore
print("🧪 Test inizializzazione Whisper...")
if HF_AVAILABLE and AUDIO_AVAILABLE:
    try:
        # Inizializza con modello small (buon compromesso velocità/qualità)
        whisper_transcriber = HuggingFaceWhisperTranscriber("openai/whisper-small")
        print("✅ Trascrittore Whisper inizializzato con successo")
        
        # Mostra modelli disponibili
        whisper_transcriber.list_available_models()
        
    except Exception as e:
        print(f"❌ Errore inizializzazione: {e}")
        whisper_transcriber = None
else:
    print("⚠️ Trascrittore non disponibile - installa le dipendenze")
    whisper_transcriber = None

## 3. Audio File Management

Implementiamo la gestione dei file audio con validazione e supporto per diversi formati.

In [ ]:
# Step 3: Preparazione File Audio
print("📂 STEP 3: Preparazione File Audio")
print("-" * 50)

import librosa
import soundfile as sf
from pathlib import Path

# Cartella audio di esempio
audio_folder = Path("audio_samples")
audio_folder.mkdir(exist_ok=True)

# Funzioni utility per gestione audio
def validate_audio_file(file_path: Path):
    """Valida un file audio."""
    try:
        # Controlla se il file esiste
        if not file_path.exists():
            return {"valid": False, "error": "File non trovato"}
        
        # Verifica dimensione (max 100MB)
        max_size_mb = 100
        size_mb = file_path.stat().st_size / (1024 * 1024)
        if size_mb > max_size_mb:
            return {"valid": False, "error": f"File troppo grande: {size_mb:.1f}MB (max {max_size_mb}MB)"}
        
        # Verifica formato supportato
        supported_formats = ['.mp3', '.wav', '.m4a', '.flac', '.ogg', '.webm']
        if file_path.suffix.lower() not in supported_formats:
            return {"valid": False, "error": f"Formato non supportato: {file_path.suffix}"}
        
        # Test caricamento con librosa
        try:
            audio, sr = librosa.load(str(file_path), sr=None, duration=1)  # Solo 1 secondo per test
            duration = librosa.get_duration(path=str(file_path))
            
            return {
                "valid": True,
                "size_mb": round(size_mb, 2),
                "sample_rate": sr,
                "duration_seconds": round(duration, 2),
                "format": file_path.suffix.lower()
            }
        except Exception as audio_error:
            return {"valid": False, "error": f"Errore caricamento audio: {str(audio_error)}"}
            
    except Exception as e:
        return {"valid": False, "error": f"Errore validazione: {str(e)}"}

print("✅ Funzioni di validazione audio caricate")
print("\n📋 Istruzioni:")
print("1. Posiziona i tuoi file audio nella cartella 'audio_samples/'")
print("2. Formati supportati: MP3, WAV, M4A, FLAC, OGG")
print("3. Dimensione massima: 100MB per file")
print("4. Per ottimali risultati, usa audio con voce chiara e poco rumore di fondo")

## 4. Create Audio File Reader Function

Implementiamo le funzioni per leggere e processare i file audio dalla cartella specificata.

In [ ]:
# Step 4: Trascrizione Audio con Hugging Face Whisper
print("🎤 STEP 4: Trascrizione Audio")
print("-" * 50)

# Test di trascrizione con il nostro modello
transcriber = HuggingFaceWhisperTranscriber("openai/whisper-small")

# Esempio di trascrizione di un file audio
def transcribe_demo_file():
    """Demo di trascrizione."""
    # Cerca file nella cartella audio_samples
    audio_files = list(audio_folder.glob("*.wav")) + list(audio_folder.glob("*.mp3"))
    
    if not audio_files:
        print("⚠️ Nessun file audio trovato nella cartella audio_samples/")
        print("   Carica un file audio per testare la trascrizione")
        return None
    
    # Prendi il primo file disponibile
    audio_file = audio_files[0]
    print(f"🎵 Elaborazione file: {audio_file.name}")
    
    # Valida il file
    validation = validate_audio_file(audio_file)
    if not validation["valid"]:
        print(f"❌ File non valido: {validation['error']}")
        return None
    
    print(f"✅ File validato - Dimensione: {validation['size_mb']}MB, Durata: {validation['duration_seconds']}s")
    
    # Trascrivi
    print("🤗 Avvio trascrizione...")
    start_time = time.time()
    
    result = transcriber.transcribe_audio(audio_file, language="italian")
    
    if result["success"]:
        data = result["data"]
        elapsed = time.time() - start_time
        
        print(f"✅ Trascrizione completata in {elapsed:.2f} secondi")
        print(f"📝 Parole trovate: {data['word_count']}")
        print(f"⚡ Velocità: {data['word_count']/elapsed:.1f} parole/sec")
        print("\n📄 Trascrizione:")
        print("-" * 40)
        print(data["text"])
        print("-" * 40)
        
        return data["text"]
    else:
        print(f"❌ Errore trascrizione: {result['error']}")
        return None

# Test del sistema
print("🧪 Test del sistema di trascrizione...")
try:
    transcript = transcribe_demo_file()
    if transcript:
        print("✅ Sistema di trascrizione funzionante!")
    else:
        print("⚠️ Carica un file audio nella cartella audio_samples/ per testare")
except Exception as e:
    print(f"❌ Errore durante test trascrizione: {e}")
    transcript = None

## 5. Implement Whisper Speech-to-Text

Implementiamo le funzioni per utilizzare OpenAI Whisper per la trascrizione audio.

In [ ]:
# Step 5: Generatore di Minute con Modello Locale (Hugging Face)
print("📋 STEP 5: Generazione Minute con Modello Locale")
print("-" * 60)

class HuggingFaceMinutesGenerator:
    """Generatore di minute utilizzando modelli Hugging Face locali."""
    
    def __init__(self, model_name: str = "microsoft/DialoGPT-medium"):
        if not HF_AVAILABLE:
            raise ImportError("Transformers non disponibile")
            
        self.model_name = model_name
        self.device = DEVICE
        self.tokenizer = None
        self.model = None
        self.text_generator = None
        
        # Prompt templates per diversi tipi di minute
        self.prompt_templates = {
            "meeting": """Analizza questa trascrizione di meeting e crea minute strutturate in italiano:

TRASCRIZIONE:
{transcript}

MINUTE STRUTTURATE:
## Sommario Esecutivo
[Breve riassunto dei punti principali]

## Partecipanti
{participants}

## Argomenti Discussi
[Punti principali trattati]

## Decisioni Prese
[Decisioni formali e approvazioni]

## Azioni e Responsabilità  
[Task assegnati con responsabili]

## Prossimi Passi
[Follow-up necessari]
""",
            
            "simple": """Riassumi questa trascrizione in minute chiare e concise:

TRASCRIZIONE: {transcript}

RIASSUNTO:""",
            
            "structured": """Crea minute professionali da questa trascrizione:

CONTESTO: Meeting del {date} - {title}
PARTECIPANTI: {participants}

TRASCRIZIONE:
{transcript}

MINUTE:"""
        }
        
        self._load_model()
    
    def _load_model(self):
        """Carica il modello per la generazione di testo."""
        try:
            print(f"🤗 Caricamento modello generativo: {self.model_name}")
            start_time = time.time()
            
            # Usa un modello più leggero per la generazione di testo
            if "DialoGPT" in self.model_name:
                # Per modelli conversazionali
                self.tokenizer = AutoTokenizer.from_pretrained(
                    self.model_name, 
                    cache_dir=str(MODELS_CACHE_DIR)
                )
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.model_name,
                    cache_dir=str(MODELS_CACHE_DIR),
                    torch_dtype=TORCH_DTYPE,
                    device_map="auto" if DEVICE == "cuda" else None
                )
                
                # Aggiungi pad token se mancante
                if self.tokenizer.pad_token is None:
                    self.tokenizer.pad_token = self.tokenizer.eos_token
                    
            else:
                # Pipeline più generale per text generation
                self.text_generator = pipeline(
                    "text-generation",
                    model=self.model_name,
                    tokenizer=self.model_name,
                    device=0 if DEVICE == "cuda" else -1,
                    torch_dtype=TORCH_DTYPE,
                    cache_dir=str(MODELS_CACHE_DIR)
                )
            
            load_time = time.time() - start_time
            print(f"✅ Modello generativo caricato in {load_time:.2f} secondi")
            
        except Exception as e:
            print(f"❌ Errore caricamento modello: {e}")
            print("🔄 Fallback a generazione template-based...")
            self.model = None
            self.tokenizer = None
            self.text_generator = None
    
    def generate_minutes_local(self, transcript: str, meeting_context: Dict[str, Any] = None) -> Dict[str, Any]:
        """Genera minute usando il modello locale o template."""
        try:
            if meeting_context is None:
                meeting_context = {
                    "title": "Meeting",
                    "date": datetime.now().strftime("%Y-%m-%d"),
                    "participants": "Partecipanti del meeting"
                }
            
            # Se il modello è disponibile, usa la generazione AI
            if self.model and self.tokenizer:
                return self._generate_with_model(transcript, meeting_context)
            elif self.text_generator:
                return self._generate_with_pipeline(transcript, meeting_context)
            else:
                return self._generate_with_template(transcript, meeting_context)
                
        except Exception as e:
            print(f"❌ Errore generazione minute: {e}")
            # Fallback a template
            return self._generate_with_template(transcript, meeting_context)
    
    def _generate_with_model(self, transcript: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """Generazione con modello conversazionale."""
        try:
            # Prepara prompt
            prompt = self.prompt_templates["structured"].format(
                transcript=transcript[:2000],  # Limita lunghezza
                **context
            )
            
            # Tokenizza
            inputs = self.tokenizer.encode(prompt + self.tokenizer.eos_token, return_tensors="pt")
            if DEVICE == "cuda":
                inputs = inputs.to("cuda")
            
            # Genera
            with torch.no_grad():
                outputs = self.model.generate(
                    inputs,
                    max_length=inputs.shape[1] + 500,
                    num_return_sequences=1,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=self.tokenizer.eos_token_id
                )
            
            # Decodifica
            generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            minutes_text = generated[len(prompt):].strip()
            
            return {
                "success": True,
                "minutes": {
                    "full_text": minutes_text,
                    "generated_at": datetime.now().isoformat(),
                    "word_count": len(minutes_text.split()),
                    "method": "local_ai_model",
                    "model_used": self.model_name
                },
                "error": None
            }
            
        except Exception as e:
            print(f"❌ Errore generazione AI: {e}")
            return self._generate_with_template(transcript, context)
    
    def _generate_with_pipeline(self, transcript: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """Generazione con pipeline text-generation."""
        try:
            prompt = f"Riassumi questo meeting in minute strutturate:\n\n{transcript[:1500]}\n\nMinute:"
            
            result = self.text_generator(
                prompt,
                max_length=len(prompt.split()) + 300,
                temperature=0.7,
                do_sample=True,
                num_return_sequences=1
            )
            
            minutes_text = result[0]["generated_text"][len(prompt):].strip()
            
            return {
                "success": True,
                "minutes": {
                    "full_text": minutes_text,
                    "generated_at": datetime.now().isoformat(),
                    "word_count": len(minutes_text.split()),
                    "method": "pipeline_generation",
                    "model_used": self.model_name
                },
                "error": None
            }
            
        except Exception as e:
            print(f"❌ Errore pipeline: {e}")
            return self._generate_with_template(transcript, context)
    
    def _generate_with_template(self, transcript: str, context: Dict[str, Any]) -> Dict[str, Any]:
        """Fallback: generazione basata su template."""
        try:
            # Analisi semplice del transcript
            sentences = [s.strip() for s in transcript.split('.') if s.strip()]
            word_count = len(transcript.split())
            
            # Template-based minutes
            minutes_text = f"""# Minute del Meeting

**Titolo:** {context.get('title', 'Meeting')}
**Data:** {context.get('date', datetime.now().strftime('%Y-%m-%d'))}
**Partecipanti:** {context.get('participants', 'Non specificati')}

## Sommario Esecutivo
Durante il meeting sono stati discussi {len(sentences)} punti principali per un totale di circa {word_count} parole di conversazione.

## Contenuto Principale
{transcript[:1000]}{'...' if len(transcript) > 1000 else ''}

## Punti Chiave Identificati
- Discussione su {sentences[0][:100] if sentences else 'argomenti vari'}
- Approfondimenti su temi strategici
- Coordinamento attività future

## Azioni da Intraprendere
- Seguire le decisioni emerse durante la discussione
- Pianificare i prossimi incontri
- Documentare i progressi

## Note
Queste minute sono state generate automaticamente utilizzando analisi testuale.
Per maggiore precisione, rivedere la trascrizione completa.

---
*Generato automaticamente il {datetime.now().strftime('%Y-%m-%d alle %H:%M')}*
"""
            
            return {
                "success": True,
                "minutes": {
                    "full_text": minutes_text,
                    "generated_at": datetime.now().isoformat(),
                    "word_count": len(minutes_text.split()),
                    "method": "template_based",
                    "model_used": "template_generator"
                },
                "error": None
            }
            
        except Exception as e:
            return {
                "success": False,
                "minutes": None,
                "error": str(e)
            }

# Test del generatore di minute
print("🧪 Test inizializzazione generatore minute...")
if HF_AVAILABLE:
    try:
        # Inizializza generatore (prova diversi modelli)
        try:
            minutes_generator = HuggingFaceMinutesGenerator("microsoft/DialoGPT-medium")
        except:
            print("⚠️ DialoGPT non disponibile, uso fallback template")
            minutes_generator = HuggingFaceMinutesGenerator("gpt2")  # Fallback più leggero
        
        print("✅ Generatore di minute inizializzato")
        
        # Test con testo di esempio
        test_transcript = """
        Buongiorno a tutti. Iniziamo il meeting di oggi con la revisione del progetto Alpha.
        Il progetto procede secondo i tempi previsti. Mario, puoi aggiornarci sui progressi?
        Certamente. Abbiamo completato la fase di sviluppo e stiamo iniziando i test.
        La consegna è prevista per la prossima settimana come programmato.
        Ottimo. Laura, come procede il budget?
        Il budget è sotto controllo. Abbiamo utilizzato il 75% delle risorse allocate.
        Bene. Per il prossimo meeting, prepariamo la presentazione finale per il cliente.
        """
        
        test_context = {
            "title": "Revisione Progetto Alpha", 
            "date": "2025-12-05",
            "participants": "Mario, Laura, Team Lead"
        }
        
        print("\n🎯 Test generazione minute...")
        result = minutes_generator.generate_minutes_local(test_transcript, test_context)
        
        if result["success"]:
            print(f"✅ Minute generate con metodo: {result['minutes']['method']}")
            print(f"📝 Parole generate: {result['minutes']['word_count']}")
        else:
            print(f"❌ Errore test: {result['error']}")
        
    except Exception as e:
        print(f"❌ Errore inizializzazione generatore: {e}")
        minutes_generator = None
else:
    print("⚠️ Generatore non disponibile - installa transformers")
    minutes_generator = None

## 6. Integrate DeepSeek R1 for Minutes Generation

Implementiamo l'integrazione con DeepSeek R1 per generare minute strutturate dalla trascrizione.

In [ ]:
# Step 6: Pipeline Completa Solo Hugging Face
print("🔄 STEP 6: Pipeline Completa (100% Locale)")
print("-" * 60)

def process_audio_to_minutes_local(
    audio_file_path: Path, 
    meeting_context: Dict[str, Any] = None,
    whisper_model: str = "openai/whisper-small",
    generation_model: str = "microsoft/DialoGPT-medium"
) -> Dict[str, Any]:
    """
    Pipeline completa utilizzando solo modelli Hugging Face locali.
    
    Args:
        audio_file_path: Percorso al file audio
        meeting_context: Informazioni sul meeting
        whisper_model: Modello Whisper da utilizzare
        generation_model: Modello per generazione minute
    
    Returns:
        Dict con risultati completi della pipeline
    """
    results = {
        "success": False,
        "transcript": None,
        "minutes": None,
        "processing_time": 0,
        "model_info": {},
        "errors": []
    }
    
    pipeline_start = time.time()
    
    try:
        print(f"🚀 Avvio pipeline completa per: {Path(audio_file_path).name}")
        
        # Step 1: Validazione file
        print("🔍 Step 1: Validazione file audio")
        validation = validate_audio_file(Path(audio_file_path))
        
        if not validation["valid"]:
            results["errors"].append(f"File non valido: {validation['error']}")
            return results
        
        print(f"✅ File validato: {validation['size_mb']}MB, {validation['duration_seconds']}s")
        results["model_info"]["audio_info"] = validation
        
        # Step 2: Trascrizione con Whisper (Hugging Face)
        print(f"🎤 Step 2: Trascrizione con {whisper_model}")
        
        # Inizializza o riusa trascrittore
        if 'whisper_transcriber' not in globals() or whisper_transcriber is None:
            transcriber = HuggingFaceWhisperTranscriber(whisper_model)
        else:
            transcriber = whisper_transcriber
            if transcriber.model_name != whisper_model:
                transcriber.change_model(whisper_model)
        
        transcript_result = transcriber.transcribe_audio(
            Path(audio_file_path), 
            language="italian"
        )
        
        if not transcript_result["success"]:
            results["errors"].append(f"Errore trascrizione: {transcript_result['error']}")
            return results
        
        results["transcript"] = transcript_result["data"]
        results["model_info"]["whisper_model"] = whisper_model
        print(f"✅ Trascrizione: {transcript_result['data']['word_count']} parole")
        
        # Step 3: Generazione minute con modello locale
        print(f"📋 Step 3: Generazione minute con {generation_model}")
        
        # Inizializza o riusa generatore
        try:
            if 'minutes_generator' not in globals() or minutes_generator is None:
                generator = HuggingFaceMinutesGenerator(generation_model)
            else:
                generator = minutes_generator
        except Exception as e:
            print(f"⚠️ Errore caricamento {generation_model}: {e}")
            print("🔄 Fallback a generazione template")
            generator = HuggingFaceMinutesGenerator("template")  # Template fallback
        
        # Contesto di default
        if meeting_context is None:
            meeting_context = {
                "title": f"Meeting Audio {datetime.now().strftime('%Y%m%d')}",
                "date": datetime.now().strftime("%Y-%m-%d"),
                "participants": "Partecipanti automaticamente rilevati"
            }
        
        minutes_result = generator.generate_minutes_local(
            transcript_result["data"]["text"], 
            meeting_context
        )
        
        if minutes_result["success"]:
            results["minutes"] = minutes_result["minutes"]
            results["model_info"]["generation_model"] = generation_model
            results["model_info"]["generation_method"] = minutes_result["minutes"]["method"]
            print(f"✅ Minute: {minutes_result['minutes']['word_count']} parole")
        else:
            results["errors"].append(f"Errore generazione: {minutes_result['error']}")
        
        # Step 4: Salvataggio risultati
        print("💾 Step 4: Salvataggio risultati")
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
        # Crea cartella output
        output_dir = PROJECT_PATH / "output"
        output_dir.mkdir(exist_ok=True)
        
        # Salva trascrizione
        transcript_file = output_dir / f"transcript_{timestamp}.txt"
        with open(transcript_file, 'w', encoding='utf-8') as f:
            f.write(f"Trascrizione Audio - {meeting_context.get('title', 'Meeting')}\n")
            f.write(f"Data: {meeting_context.get('date')}\n")
            f.write(f"Modello Whisper: {whisper_model}\n")
            f.write(f"Tempo elaborazione: {transcript_result['data']['processing_time_seconds']:.2f}s\n")
            f.write(f"Parole: {transcript_result['data']['word_count']}\n")
            f.write(f"Velocità: {transcript_result['data']['words_per_second']:.1f} parole/sec\n")
            f.write("-" * 80 + "\n\n")
            f.write(transcript_result["data"]["text"])
        
        print(f"📄 Trascrizione salvata: {transcript_file.name}")
        
        # Salva minute se disponibili
        if results["minutes"]:
            minutes_file = output_dir / f"minutes_{timestamp}.md"
            with open(minutes_file, 'w', encoding='utf-8') as f:
                f.write(f"# Minute - {meeting_context.get('title', 'Meeting')}\n\n")
                f.write(f"**Data:** {meeting_context.get('date')}\n")
                f.write(f"**Partecipanti:** {meeting_context.get('participants')}\n")
                f.write(f"**Generato con:** {results['model_info']['generation_method']}\n")
                f.write(f"**Modello utilizzato:** {generation_model}\n")
                f.write(f"**Timestamp:** {results['minutes']['generated_at']}\n\n")
                f.write("---\n\n")
                f.write(results["minutes"]["full_text"])
            
            print(f"📋 Minute salvate: {minutes_file.name}")
        
        # Salva metadati
        metadata_file = output_dir / f"metadata_{timestamp}.json"
        metadata = {
            "processing_info": {
                "total_time": time.time() - pipeline_start,
                "timestamp": datetime.now().isoformat(),
                "success": True
            },
            "models_used": results["model_info"],
            "meeting_context": meeting_context,
            "files_generated": [
                transcript_file.name,
                minutes_file.name if results["minutes"] else None,
                metadata_file.name
            ]
        }
        
        with open(metadata_file, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        
        print(f"📊 Metadati salvati: {metadata_file.name}")
        
        results["success"] = True
        print(f"🎉 Pipeline completata con successo!")
        
    except Exception as e:
        error_msg = f"Errore pipeline: {str(e)}"
        results["errors"].append(error_msg)
        print(f"❌ {error_msg}")
        print(f"🔍 Traceback: {traceback.format_exc()}")
    
    finally:
        results["processing_time"] = time.time() - pipeline_start
    
    return results

def demo_complete_pipeline():
    """Demo della pipeline completa con modelli locali."""
    print("🎬 DEMO: Pipeline Completa Locale")
    print("=" * 50)
    
    # Cerca file audio disponibili
    audio_files = list(audio_folder.glob("*.wav")) + list(audio_folder.glob("*.mp3"))
    
    if not audio_files:
        print("⚠️ Nessun file audio trovato in audio_samples/")
        print("   Carica un file audio per testare la pipeline")
        return None
    
    # Usa il primo file disponibile
    test_file = audio_files[0]
    print(f"📁 File di test: {test_file.name}")
    
    # Contesto meeting di esempio
    demo_context = {
        "title": "Demo Meeting - Pipeline Locale",
        "date": datetime.now().strftime("%Y-%m-%d"),
        "participants": "Sistema di Test, AI Assistant, Demo User"
    }
    
    # Esegui pipeline con modelli ottimizzati
    try:
        print("⚡ Configurazione modelli:")
        print("   - Whisper: openai/whisper-small (compromesso velocità/qualità)")
        print("   - Generazione: microsoft/DialoGPT-medium (locale)")
        print()
        
        result = process_audio_to_minutes_local(
            test_file, 
            demo_context,
            whisper_model="openai/whisper-small",
            generation_model="microsoft/DialoGPT-medium"
        )
        
        if result["success"]:
            print("\n🎉 RISULTATI DEMO:")
            print(f"⏱️  Tempo totale: {result['processing_time']:.2f}s")
            
            if result["transcript"]:
                t = result["transcript"]
                print(f"🎤 Trascrizione: {t['word_count']} parole ({t['words_per_second']:.1f} p/s)")
                print(f"   Modello: {result['model_info']['whisper_model']}")
            
            if result["minutes"]:
                m = result["minutes"]
                print(f"📋 Minute: {m['word_count']} parole")
                print(f"   Metodo: {m['method']}")
                print(f"   Modello: {result['model_info'].get('generation_model', 'template')}")
            
            print("\n✅ Demo completata - tutti i file sono in output/")
            return result
            
        else:
            print("❌ Errori nella demo:")
            for error in result["errors"]:
                print(f"   - {error}")
            return None
            
    except Exception as e:
        print(f"❌ Errore demo: {e}")
        return None

# Esegui demo automatica
print("🚀 Avvio demo pipeline locale...")
if HF_AVAILABLE and AUDIO_AVAILABLE:
    demo_result = demo_complete_pipeline()
    
    if demo_result:
        print(f"\n🎯 PIPELINE LOCALE PRONTA!")
        print("Caratteristiche:")
        print("   ✅ Nessuna API esterna richiesta")
        print("   ✅ Elaborazione completamente offline")
        print("   ✅ Modelli Hugging Face ottimizzati")
        print("   ✅ Cache locale per velocità")
        print("   ✅ Fallback template garantiti")
        print(f"   ✅ Device: {DEVICE}")
else:
    print("⚠️ Demo non disponibile - installa le dipendenze richieste")
    print("   pip install transformers torch librosa soundfile")

## 7. Build Streamlit UI Components

Creiamo i componenti dell'interfaccia utente per l'app Streamlit.

In [ ]:
class StreamlitUI:
    """Componenti UI per l'applicazione Streamlit."""
    
    @staticmethod
    def setup_page_config():
        """Configura la pagina Streamlit."""
        st.set_page_config(
            page_title="🎤 Speech to Minutes App",
            page_icon="🎤",
            layout="wide",
            initial_sidebar_state="expanded"
        )
        
        # CSS personalizzato
        st.markdown(\"\"\"
        <style>
        .main-header {
            background: linear-gradient(90deg, #667eea 0%, #764ba2 100%);
            padding: 1rem;
            border-radius: 10px;
            margin-bottom: 2rem;
        }
        .main-header h1 {
            color: white;
            text-align: center;
            margin: 0;
        }
        .metric-container {
            background: #f0f2f6;
            padding: 1rem;
            border-radius: 8px;
            margin: 0.5rem 0;
        }
        .success-message {
            background: #d4edda;
            color: #155724;
            padding: 1rem;
            border-radius: 8px;
            border-left: 5px solid #28a745;
        }
        .error-message {
            background: #f8d7da;
            color: #721c24;
            padding: 1rem;
            border-radius: 8px;
            border-left: 5px solid #dc3545;
        }
        </style>
        \"\"\", unsafe_allow_html=True)
    
    @staticmethod
    def render_header():
        """Renderizza l'header dell'applicazione."""
        st.markdown(\"\"\"
        <div class="main-header">
            <h1>🎤 Speech to Minutes App</h1>
            <p style="text-align: center; color: white; margin: 0;">
                Trasforma i tuoi file audio in minute di meeting professionali
            </p>
        </div>
        \"\"\", unsafe_allow_html=True)
    
    @staticmethod
    def render_sidebar():
        """Renderizza la sidebar con configurazioni."""
        st.sidebar.title("⚙️ Configurazioni")
        
        # Configurazioni audio
        st.sidebar.subheader("🎵 Audio Settings")
        
        # Modello Whisper
        whisper_models = ["tiny", "base", "small", "medium", "large"]
        selected_model = st.sidebar.selectbox(
            "Modello Whisper",
            whisper_models,
            index=whisper_models.index(WHISPER_MODEL) if WHISPER_MODEL in whisper_models else 2,
            help="Modelli più grandi = maggiore accuratezza ma tempi più lunghi"
        )
        
        # Lingua di trascrizione
        languages = {
            "Italiano": "it",
            "Inglese": "en",
            "Francese": "fr",
            "Spagnolo": "es",
            "Tedesco": "de",
            "Auto-detect": None
        }
        selected_language = st.sidebar.selectbox(
            "Lingua audio",
            list(languages.keys()),
            index=0
        )
        
        # Configurazioni meeting
        st.sidebar.subheader("📋 Meeting Info")
        meeting_title = st.sidebar.text_input(
            "Titolo Meeting",
            value="Meeting Team",
            help="Titolo del meeting per le minute"
        )
        
        meeting_date = st.sidebar.date_input(
            "Data Meeting",
            value=datetime.now().date()
        )
        
        participants = st.sidebar.text_area(
            "Partecipanti",
            value="Team Members",
            height=60,
            help="Lista dei partecipanti al meeting"
        )
        
        # Output settings
        st.sidebar.subheader("📄 Output Settings")
        include_timestamps = st.sidebar.checkbox(
            "Includi timestamp",
            value=True,
            help="Include timestamp nella trascrizione"
        )
        
        include_speaker_detection = st.sidebar.checkbox(
            "Rilevamento speaker",
            value=False,
            help="Tenta di identificare i cambi di speaker (sperimentale)"
        )
        
        return {
            "whisper_model": selected_model,
            "language": languages[selected_language],
            "meeting_title": meeting_title,
            "meeting_date": meeting_date.isoformat(),
            "participants": participants,
            "include_timestamps": include_timestamps,
            "include_speaker_detection": include_speaker_detection
        }
    
    @staticmethod
    def render_file_selector(audio_manager: AudioFileManager):
        """Renderizza il selettore di file audio."""
        st.subheader("📁 Selezione File Audio")
        
        # Opzioni per l'upload
        upload_option = st.radio(
            "Come vuoi fornire il file audio?",
            ["📂 Seleziona da cartella locale", "⬆️ Upload file"],
            horizontal=True
        )
        
        selected_file = None
        
        if upload_option == "📂 Seleziona da cartella locale":
            # Lista file dalla cartella
            audio_files = audio_manager.get_audio_files()
            
            if audio_files:
                file_options = {}
                for file_path in audio_files:
                    info = audio_manager.validate_audio_file(file_path)
                    if info["valid"]:
                        display_name = f"{file_path.name} ({info['size_mb']:.1f}MB, {info['duration_seconds']:.0f}s)"
                        file_options[display_name] = file_path
                    else:
                        display_name = f"{file_path.name} - ❌ {info['error']}"
                        file_options[display_name] = None
                
                if file_options:
                    selected_display = st.selectbox(
                        "Seleziona file audio:",
                        list(file_options.keys())
                    )
                    selected_file = file_options[selected_display]
                    
                    if selected_file:
                        # Mostra dettagli del file selezionato
                        info = audio_manager.validate_audio_file(selected_file)
                        col1, col2, col3 = st.columns(3)
                        with col1:
                            st.metric("📏 Dimensione", f"{info['size_mb']:.2f} MB")
                        with col2:
                            st.metric("⏱️ Durata", f"{info['duration_seconds']:.1f}s")
                        with col3:
                            st.metric("🎛️ Sample Rate", f"{info['sample_rate']} Hz")
                else:
                    st.warning("Nessun file audio valido trovato nella cartella.")
            else:
                st.info(f"Nessun file audio trovato in: {audio_manager.audio_folder}")
                st.info(f"Formati supportati: {', '.join(SUPPORTED_AUDIO_FORMATS)}")
        
        else:  # Upload file
            uploaded_file = st.file_uploader(
                "Carica un file audio",
                type=SUPPORTED_AUDIO_FORMATS,
                help=f"Dimensione massima: {MAX_AUDIO_FILE_SIZE_MB}MB"
            )
            
            if uploaded_file:
                # Salva il file temporaneamente
                temp_path = PROJECT_PATH / "temp_uploads"
                temp_path.mkdir(exist_ok=True)
                temp_file_path = temp_path / uploaded_file.name
                
                with open(temp_file_path, "wb") as f:
                    f.write(uploaded_file.getvalue())
                
                # Valida il file
                info = audio_manager.validate_audio_file(temp_file_path)
                if info["valid"]:
                    selected_file = temp_file_path
                    
                    col1, col2, col3 = st.columns(3)
                    with col1:
                        st.metric("📏 Dimensione", f"{info['size_mb']:.2f} MB")
                    with col2:
                        st.metric("⏱️ Durata", f"{info['duration_seconds']:.1f}s")
                    with col3:
                        st.metric("🎛️ Sample Rate", f"{info['sample_rate']} Hz")
                else:
                    st.error(f"❌ File non valido: {info['error']}")
        
        return selected_file
    
    @staticmethod
    def render_processing_status():
        """Renderizza la sezione di stato del processing."""
        if "processing_status" not in st.session_state:
            st.session_state.processing_status = {
                "transcription": {"status": "pending", "message": "In attesa di avvio"},
                "minutes_generation": {"status": "pending", "message": "In attesa di avvio"},
                "current_step": 0
            }
        
        st.subheader("⚡ Stato Elaborazione")
        
        # Progress bar globale
        progress_steps = ["🎵 Trascrizione Audio", "📝 Generazione Minute"]
        current_step = st.session_state.processing_status["current_step"]
        
        progress_bar = st.progress(0)
        status_text = st.empty()
        
        # Status per ogni step
        col1, col2 = st.columns(2)
        
        with col1:
            transcription_status = st.session_state.processing_status["transcription"]
            status_icon = {"pending": "⏳", "processing": "⚡", "completed": "✅", "error": "❌"}
            st.write(f"{status_icon[transcription_status['status']]} **Trascrizione**: {transcription_status['message']}")
        
        with col2:
            minutes_status = st.session_state.processing_status["minutes_generation"]
            st.write(f"{status_icon[minutes_status['status']]} **Generazione Minute**: {minutes_status['message']}")
        
        return progress_bar, status_text
    
    @staticmethod
    def render_results(transcription_result: Dict, minutes_result: Dict):
        """Renderizza i risultati della trascrizione e delle minute."""
        st.subheader("📊 Risultati")
        
        # Tab per organizzare i risultati
        tab1, tab2, tab3, tab4 = st.tabs(["📝 Minute", "📄 Trascrizione", "⏱️ Transcript con Timestamp", "📈 Statistiche"])
        
        with tab1:
            if minutes_result and minutes_result.get("success"):
                minutes = minutes_result["minutes"]
                st.markdown("### 📋 Minute del Meeting")
                st.markdown(minutes["full_text"])
                
                # Download button per le minute
                st.download_button(
                    label="💾 Scarica Minute (MD)",
                    data=minutes["full_text"],
                    file_name=f"minutes_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md",
                    mime="text/markdown"
                )
            else:
                st.error("❌ Errore nella generazione delle minute")
        
        with tab2:
            if transcription_result and transcription_result.get("success"):
                data = transcription_result["data"]
                st.markdown("### 📝 Trascrizione Completa")
                st.text_area("Testo trascritto:", data["text"], height=300)
                
                # Download button per la trascrizione
                st.download_button(
                    label="💾 Scarica Trascrizione (TXT)",
                    data=data["text"],
                    file_name=f"transcript_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt",
                    mime="text/plain"
                )
            else:
                st.error("❌ Errore nella trascrizione")
        
        with tab3:
            if transcription_result and transcription_result.get("success"):
                data = transcription_result["data"]
                if "segments" in data and data["segments"]:
                    st.markdown("### ⏰ Trascrizione con Timestamp")
                    
                    # Crea transcript formattato
                    transcriber = WhisperTranscriber()  # Istanza temporanea
                    formatted_transcript = transcriber.get_transcript_with_timestamps(data["segments"])
                    
                    st.text_area("Transcript con timestamp:", formatted_transcript, height=400)
                    
                    # Download button
                    st.download_button(
                        label="💾 Scarica Transcript con Timestamp",
                        data=formatted_transcript,
                        file_name=f"transcript_timestamps_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt",
                        mime="text/plain"
                    )
                else:
                    st.info("⚠️ Timestamp non disponibili")
            else:
                st.error("❌ Dati di trascrizione non disponibili")
        
        with tab4:
            if transcription_result and transcription_result.get("success"):
                data = transcription_result["data"]
                
                st.markdown("### 📈 Statistiche Elaborazione")
                
                col1, col2, col3, col4 = st.columns(4)
                
                with col1:
                    st.metric(
                        "⏱️ Durata Audio",
                        f"{data.get('audio_duration', 0):.1f}s"
                    )
                
                with col2:
                    st.metric(
                        "🚀 Tempo Elaborazione",
                        f"{data.get('processing_time_seconds', 0):.1f}s"
                    )
                
                with col3:
                    st.metric(
                        "📝 Parole Trascritte",
                        f"{data.get('word_count', 0)}"
                    )
                
                with col4:
                    processing_ratio = (data.get('processing_time_seconds', 1) / max(data.get('audio_duration', 1), 1))
                    st.metric(
                        "⚡ Velocità (x tempo reale)",
                        f"{processing_ratio:.2f}x"
                    )
                
                if minutes_result and minutes_result.get("success"):
                    minutes = minutes_result["minutes"]
                    st.markdown("### 📊 Statistiche Minute")
                    
                    col1, col2 = st.columns(2)
                    with col1:
                        st.metric("📄 Parole Minute", minutes.get("word_count", 0))
                    with col2:
                        ratio = (minutes.get("word_count", 0) / max(data.get("word_count", 1), 1)) * 100
                        st.metric("📉 Compressione", f"{ratio:.1f}%")

print("🎨 Streamlit UI Components creati!")
print("✅ Pronto per la creazione dell'applicazione principale")

## 8. Create Main Application Logic

Implementiamo la logica principale dell'applicazione Streamlit che orchestra tutto il pipeline.

In [ ]:
def main_streamlit_app():
    """Applicazione Streamlit principale per Speech to Minutes."""
    
    # Setup pagina
    StreamlitUI.setup_page_config()
    StreamlitUI.render_header()
    
    # Sidebar con configurazioni
    config = StreamlitUI.render_sidebar()
    
    # Inizializza i componenti principali
    audio_manager = AudioFileManager(PROJECT_PATH / "audio")
    
    # Crea le colonne principale
    col1, col2 = st.columns([2, 1])
    
    with col1:
        # Selezione file
        selected_file = StreamlitUI.render_file_selector(audio_manager)
        
        # Pulsante per avviare l'elaborazione
        if selected_file and st.button("🚀 Avvia Elaborazione", type="primary", use_container_width=True):
            
            # Inizializza il transcriber con il modello selezionato
            try:
                with st.spinner("🤖 Caricamento modello Whisper..."):
                    transcriber = WhisperTranscriber(
                        model_name=config["whisper_model"],
                        device=WHISPER_DEVICE
                    )
                
                # Inizializza il generatore di minute
                minutes_generator = DeepSeekMinutesGenerator()
                
                # Progress tracking
                progress_bar, status_text = StreamlitUI.render_processing_status()
                
                # STEP 1: Trascrizione
                st.session_state.processing_status["transcription"]["status"] = "processing"
                st.session_state.processing_status["transcription"]["message"] = "Trascrizione in corso..."
                st.session_state.processing_status["current_step"] = 1
                
                progress_bar.progress(0.1)
                status_text.text("🎵 Avvio trascrizione audio...")
                
                transcription_result = transcriber.transcribe_audio(
                    selected_file,
                    language=config["language"]
                )
                
                if transcription_result["success"]:
                    st.session_state.processing_status["transcription"]["status"] = "completed"
                    st.session_state.processing_status["transcription"]["message"] = "Trascrizione completata"
                    
                    progress_bar.progress(0.5)
                    status_text.text("✅ Trascrizione completata, generazione minute...")
                    
                    # STEP 2: Generazione minute
                    st.session_state.processing_status["minutes_generation"]["status"] = "processing"
                    st.session_state.processing_status["minutes_generation"]["message"] = "Generazione minute in corso..."
                    st.session_state.processing_status["current_step"] = 2
                    
                    meeting_context = {
                        "title": config["meeting_title"],
                        "date": config["meeting_date"],
                        "participants": config["participants"]
                    }
                    
                    minutes_result = minutes_generator.generate_minutes(
                        transcription_result["data"]["text"],
                        meeting_context
                    )
                    
                    if minutes_result["success"]:
                        st.session_state.processing_status["minutes_generation"]["status"] = "completed"
                        st.session_state.processing_status["minutes_generation"]["message"] = "Minute generate"
                        
                        progress_bar.progress(1.0)
                        status_text.text("🎉 Elaborazione completata con successo!")
                        
                        # Salva i risultati nel session state per la visualizzazione
                        st.session_state.last_transcription_result = transcription_result
                        st.session_state.last_minutes_result = minutes_result
                        
                        # Messaggio di successo
                        st.success("🎉 Elaborazione completata con successo!")
                        
                        # Mostra risultati
                        StreamlitUI.render_results(transcription_result, minutes_result)
                        
                    else:
                        st.session_state.processing_status["minutes_generation"]["status"] = "error"
                        st.session_state.processing_status["minutes_generation"]["message"] = f"Errore: {minutes_result['error']}"
                        
                        st.error(f"❌ Errore nella generazione delle minute: {minutes_result['error']}")
                        
                        # Mostra comunque la trascrizione
                        if transcription_result["success"]:
                            st.info("📝 Trascrizione disponibile:")
                            with st.expander("Visualizza trascrizione"):
                                st.text_area("Testo trascritto:", transcription_result["data"]["text"], height=200)
                        
                else:
                    st.session_state.processing_status["transcription"]["status"] = "error"
                    st.session_state.processing_status["transcription"]["message"] = f"Errore: {transcription_result['error']}"
                    
                    st.error(f"❌ Errore nella trascrizione: {transcription_result['error']}")
                
            except Exception as e:
                logger.error(f"Errore nell'applicazione: {e}")
                st.error(f"❌ Errore critico: {str(e)}")
                st.error("Dettagli: " + traceback.format_exc())
    
    with col2:
        # Informazioni e statistiche
        st.subheader("ℹ️ Informazioni")
        
        # Stato configurazione
        if DEEPSEEK_API_KEY != "your_deepseek_api_key_here":
            st.success("✅ API DeepSeek configurata")
        else:
            st.warning("⚠️ Configura API DeepSeek")
        
        # Modello selezionato
        st.info(f"🤖 Modello Whisper: {config['whisper_model']}")
        
        # Statistiche cartella audio
        audio_files = audio_manager.get_audio_files()
        st.info(f"📁 File audio disponibili: {len(audio_files)}")
        
        # Mostra risultati precedenti se disponibili
        if "last_transcription_result" in st.session_state and "last_minutes_result" in st.session_state:
            st.subheader("📊 Ultima Elaborazione")
            
            trans_data = st.session_state.last_transcription_result["data"]
            minutes_data = st.session_state.last_minutes_result["minutes"]
            
            st.metric("⏱️ Tempo elaborazione", f"{trans_data['processing_time_seconds']:.1f}s")
            st.metric("📝 Parole trascritte", trans_data['word_count'])
            st.metric("📄 Parole minute", minutes_data.get('word_count', 0))
            
            # Pulsante per mostrare di nuovo i risultati
            if st.button("🔄 Mostra Ultima Elaborazione"):
                StreamlitUI.render_results(
                    st.session_state.last_transcription_result,
                    st.session_state.last_minutes_result
                )
        
        # Sezione help
        st.subheader("❓ Help")
        with st.expander("Come usare l'app"):
            st.markdown(\"\"\"
            **Passaggi per utilizzare l'app:**
            
            1. 🔧 **Configura l'API DeepSeek** nel file .env
            2. 📁 **Aggiungi file audio** nella cartella 'audio/'
            3. ⚙️ **Configura le impostazioni** nella sidebar
            4. 📂 **Seleziona un file audio** dall'elenco
            5. 🚀 **Clicca "Avvia Elaborazione"**
            6. ⏳ **Attendi l'elaborazione**
            7. 📋 **Visualizza e scarica le minute**
            
            **Formati audio supportati:**
            {', '.join(SUPPORTED_AUDIO_FORMATS)}
            
            **Dimensione massima file:** {MAX_AUDIO_FILE_SIZE_MB}MB
            \"\"\")

## 9. Test the Complete Pipeline

Testiamo l'applicazione completa e creiamo il file Streamlit standalone.

In [ ]:
# Step 7: Riepilogo e Test Finali
print("🎯 STEP 7: Riepilogo Sistema Completo")
print("-" * 60)

def system_health_check():
    """Controllo completo dello stato del sistema."""
    print("🔍 SYSTEM HEALTH CHECK")
    print("-" * 30)
    
    status = {
        "dependencies": {},
        "models": {},
        "hardware": {},
        "storage": {}
    }
    
    # Check dipendenze
    print("📦 Controllo dipendenze:")
    deps = {
        "transformers": HF_AVAILABLE,
        "torch": HF_AVAILABLE,
        "librosa": AUDIO_AVAILABLE,
        "numpy": True,
        "pathlib": True
    }
    
    for dep, available in deps.items():
        icon = "✅" if available else "❌"
        print(f"   {icon} {dep}")
        status["dependencies"][dep] = available
    
    # Check hardware
    print(f"\n💻 Hardware:")
    print(f"   Device: {DEVICE}")
    if DEVICE == "cuda":
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
        status["hardware"]["gpu"] = True
    else:
        print(f"   CPU: Elaborazione su processore")
        status["hardware"]["gpu"] = False
    
    # Check modelli caricati
    print(f"\n🤖 Modelli:")
    models_status = {
        "whisper": whisper_transcriber is not None,
        "minutes_generator": minutes_generator is not None
    }
    
    for model, loaded in models_status.items():
        icon = "✅" if loaded else "❌"
        print(f"   {icon} {model}")
        status["models"][model] = loaded
    
    # Check storage
    print(f"\n💾 Storage:")
    print(f"   Cache modelli: {MODELS_CACHE_DIR}")
    print(f"   Directory output: {PROJECT_PATH / 'output'}")
    
    cache_size = sum(f.stat().st_size for f in MODELS_CACHE_DIR.rglob('*') if f.is_file()) / 1e9
    print(f"   Spazio cache: {cache_size:.2f}GB")
    status["storage"]["cache_size_gb"] = cache_size
    
    # Risultato finale
    all_deps = all(status["dependencies"].values())
    all_models = all(status["models"].values())
    
    if all_deps and all_models:
        print(f"\n🎉 SISTEMA OPERATIVO AL 100%")
        status["overall"] = "excellent"
    elif all_deps:
        print(f"\n✅ SISTEMA FUNZIONALE (alcuni modelli da inizializzare)")
        status["overall"] = "good"
    else:
        print(f"\n⚠️ SISTEMA PARZIALE (dipendenze mancanti)")
        status["overall"] = "limited"
    
    return status

def performance_benchmark():
    """Test delle performance del sistema."""
    print(f"\n⚡ BENCHMARK PERFORMANCE")
    print("-" * 30)
    
    if not (HF_AVAILABLE and AUDIO_AVAILABLE):
        print("❌ Benchmark non disponibile - dipendenze mancanti")
        return None
    
    # Test Whisper
    if whisper_transcriber:
        print("🎤 Test velocità Whisper:")
        # Simula trascrizione di 1 minuto di audio
        dummy_audio = np.random.randn(16000)  # 1 secondo di audio
        start_time = time.time()
        
        try:
            # Simula elaborazione
            test_result = whisper_transcriber.pipe(dummy_audio)
            whisper_time = time.time() - start_time
            print(f"   Tempo elaborazione 1s audio: {whisper_time:.3f}s")
            print(f"   Velocità relativa: {1/whisper_time:.1f}x")
        except:
            print(f"   Test non completato")
    
    # Test generazione
    if minutes_generator:
        print("\n📋 Test generazione minute:")
        test_text = "Questo è un test di velocità per la generazione di minute automatiche."
        start_time = time.time()
        
        try:
            result = minutes_generator.generate_minutes_local(test_text)
            gen_time = time.time() - start_time
            print(f"   Tempo generazione: {gen_time:.3f}s")
            if result["success"]:
                words = result["minutes"]["word_count"]
                print(f"   Velocità: {words/gen_time:.1f} parole/sec")
        except:
            print(f"   Test non completato")
    
    print(f"\n💡 OTTIMIZZAZIONI CONSIGLIATE:")
    if DEVICE == "cpu":
        print("   - Usa GPU per prestazioni 5-10x migliori")
    else:
        print("   ✅ GPU attiva - prestazioni ottimali")
    
    print("   - Cache modelli attiva per ricaricamenti veloci")
    print("   - Preprocessing audio ottimizzato a 16kHz")

def usage_examples():
    """Esempi di utilizzo del sistema."""
    print(f"\n📚 ESEMPI DI UTILIZZO")
    print("-" * 25)
    
    print("1️⃣ ELABORAZIONE SINGOLA:")
    print('''
    # Carica file audio
    audio_file = Path("audio_samples/meeting.wav")
    
    # Elabora con pipeline completa
    result = process_audio_to_minutes_local(
        audio_file,
        meeting_context={
            "title": "Team Meeting",
            "date": "2025-12-05",
            "participants": "Mario, Laura, Giuseppe"
        }
    )
    
    if result["success"]:
        print(f"Trascrizione: {result['transcript']['text']}")
        print(f"Minute: {result['minutes']['full_text']}")
    ''')
    
    print("\n2️⃣ ELABORAZIONE BATCH:")
    print('''
    # Elabora tutti i file in una cartella
    audio_dir = Path("meetings/")
    for audio_file in audio_dir.glob("*.wav"):
        print(f"Elaborazione {audio_file.name}...")
        result = process_audio_to_minutes_local(audio_file)
    ''')
    
    print("\n3️⃣ PERSONALIZZAZIONE MODELLI:")
    print('''
    # Modelli veloci (per test)
    result = process_audio_to_minutes_local(
        audio_file,
        whisper_model="openai/whisper-tiny",
        generation_model="template"  # Solo template
    )
    
    # Modelli di qualità (per produzione)
    result = process_audio_to_minutes_local(
        audio_file,
        whisper_model="openai/whisper-medium",
        generation_model="microsoft/DialoGPT-large"
    )
    ''')

# Esegui controlli finali
print("🚀 ESECUZIONE CONTROLLI FINALI...")

system_status = system_health_check()
performance_benchmark()
usage_examples()

print(f"\n" + "="*60)
print("🎉 SPEECH-TO-MINUTES SYSTEM READY!")
print("="*60)

print(f"\n📋 CARATTERISTICHE PRINCIPALI:")
print("   🤗 Modelli Hugging Face completamente locali")
print("   🚫 Nessuna API esterna richiesta")
print("   🔒 Privacy completa - elaborazione offline")
print("   ⚡ Ottimizzazioni GPU/CPU automatiche")
print("   💾 Cache intelligente dei modelli")
print("   📁 Output strutturato (trascrizione + minute + metadati)")
print("   🛠️ Fallback robusti garantiti")

print(f"\n🎯 COME INIZIARE:")
print("   1. Carica file audio in audio_samples/")
print("   2. Esegui: process_audio_to_minutes_local(audio_file)")
print("   3. Trova risultati in output/")

print(f"\n💡 MODELLI RACCOMANDATI:")
print("   • Test/Demo: whisper-tiny + template")
print("   • Produzione: whisper-small + DialoGPT-medium")  
print("   • Alta qualità: whisper-medium/large + modelli avanzati")

print(f"\n📊 STATUS SISTEMA: {system_status['overall'].upper()}")

if system_status['overall'] == 'excellent':
    print("🚀 Pronto per l'uso in produzione!")
elif system_status['overall'] == 'good':
    print("✅ Funzionale - inizializza i modelli per prestazioni complete")
else:
    print("⚠️ Installa le dipendenze richieste per funzionalità complete")

print(f"\n🔗 PROSSIMI PASSI:")
print("   • Testa con i tuoi file audio")
print("   • Personalizza i prompt per le tue esigenze")
print("   • Sperimenta con modelli diversi")
print("   • Integra in altre applicazioni")

print(f"\n" + "="*60)

## 🚀 Avvia l'Applicazione

Per testare l'applicazione, esegui la cella sottostante oppure usa i file standalone creati.